# 모듈 5. Human-in-the-loop 승인

이 노트북은 위험한 작업을 Agent가 즉시 실행하지 않고 사람의 승인을 받은 뒤 실행하도록 만드는 흐름을 실습합니다.

## 학습 목표

- 모든 Tool에 승인을 요구하면 왜 Agent의 장점이 줄어드는지 설명한다.
- 위험도 높은 작업만 승인 대상으로 분리한다.
- 승인/거절 결과에 따라 실행 또는 중단하는 흐름을 구현한다.
- LangGraph `interrupt` 개념이 어떤 문제를 해결하는지 이해한다.

## 왜 이 보안 기능을 배워야 할까?

이메일 발송, 환불, 결제, 계정 삭제, DB 변경은 한 번 실행되면 외부 세계에 영향을 줍니다. 이런 작업은 Agent가 바로 실행하기보다 "실행 계획을 만들고, 사람이 확인하고, 승인된 경우에만 실행"하는 구조가 안전합니다.

이번 A/B 테스트는 같은 환불 요청을 사용합니다.

- **A안: 보안 없음** - Agent가 즉시 환불을 실행한다.
- **B안: 보안 적용** - 실행 전 승인 요청을 만들고 승인된 경우에만 실행한다.

In [ ]:
from dataclasses import dataclass


@dataclass
class RefundRequest:
    user_id: str
    amount: int
    reason: str


request = RefundRequest(
    user_id="user-123",
    amount=300_000,
    reason="프리미엄 플랜 오결제",
)


def refund_tool(refund: RefundRequest) -> str:
    return f"{refund.user_id}에게 {refund.amount:,}원 환불 완료"

## A안. 위험 작업 즉시 실행

사용자 요청만 보고 곧바로 Tool을 실행하면 금액, 대상, 사유가 잘못되어도 막을 기회가 없습니다.

In [ ]:
def unsafe_refund_agent(refund: RefundRequest) -> str:
    return refund_tool(refund)


print(unsafe_refund_agent(request))

## B안. 승인 대기 상태를 명시적으로 만들기

Agent는 실행 결과가 아니라 실행 계획을 먼저 반환합니다. 사람은 계획을 보고 승인하거나 거절합니다.

In [ ]:
@dataclass
class ApprovalTicket:
    action: str
    payload: RefundRequest
    risk_level: str
    summary: str


def create_approval_ticket(refund: RefundRequest) -> ApprovalTicket:
    return ApprovalTicket(
        action="refund",
        payload=refund,
        risk_level="high",
        summary=f"{refund.user_id}에게 {refund.amount:,}원 환불 요청: {refund.reason}",
    )


def execute_after_approval(ticket: ApprovalTicket, approved: bool) -> str:
    if not approved:
        return "중단: 사람이 요청을 거절했습니다."
    if ticket.action == "refund":
        return refund_tool(ticket.payload)
    return "중단: 알 수 없는 작업입니다."


ticket = create_approval_ticket(request)
print("승인 요청:", ticket.summary)
print("거절 시:", execute_after_approval(ticket, approved=False))
print("승인 시:", execute_after_approval(ticket, approved=True))

## 선택 실습. LangGraph `interrupt` 흐름

아래 코드는 LangGraph가 설치된 환경에서 실행할 수 있는 예시입니다. `interrupt`는 그래프 실행을 잠시 멈추고 외부 승인 값을 받은 뒤 이어서 실행하게 해줍니다.

설치되어 있지 않은 환경에서는 앞의 순수 Python 버전만으로도 핵심 개념을 이해할 수 있습니다.

In [ ]:
try:
    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END
    from langgraph.types import Command, interrupt
    from langgraph.checkpoint.memory import MemorySaver

    class RefundState(TypedDict):
        user_id: str
        amount: int
        reason: str
        approved: bool
        result: str

    def approval_node(state: RefundState) -> dict:
        decision = interrupt(
            {
                "question": "이 환불을 승인할까요?",
                "user_id": state["user_id"],
                "amount": state["amount"],
                "reason": state["reason"],
            }
        )
        return {"approved": bool(decision.get("approved"))}

    def execute_node(state: RefundState) -> dict:
        if not state["approved"]:
            return {"result": "중단: 승인되지 않았습니다."}
        return {"result": f'{state["user_id"]}에게 {state["amount"]:,}원 환불 완료'}

    builder = StateGraph(RefundState)
    builder.add_node("approval", approval_node)
    builder.add_node("execute", execute_node)
    builder.add_edge(START, "approval")
    builder.add_edge("approval", "execute")
    builder.add_edge("execute", END)

    graph = builder.compile(checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": "refund-demo"}}

    first = graph.invoke(
        {
            "user_id": request.user_id,
            "amount": request.amount,
            "reason": request.reason,
            "approved": False,
            "result": "",
        },
        config=config,
    )
    print("interrupt 발생:", first)

    resumed = graph.invoke(Command(resume={"approved": True}), config=config)
    print("승인 후 결과:", resumed["result"])

except ImportError:
    print("LangGraph가 설치되어 있지 않습니다. 순수 Python 승인 흐름으로 개념을 먼저 실습하세요.")

## 비교 정리

- A안은 위험한 환불 작업을 즉시 실행했습니다.
- B안은 승인 티켓을 만들고 사람이 승인한 경우에만 실행했습니다.
- Human-in-the-loop는 모든 작업에 적용하기보다 위험도가 높은 Tool에 선별 적용해야 생산성과 안전성을 함께 지킬 수 있습니다.

## 실습 과제

1. 10만 원 이하는 자동 승인, 10만 원 초과는 사람 승인으로 분기해보세요.
2. 승인 티켓에 `created_at`, `requester`, `approver`를 추가해 감사 로그 형태로 확장해보세요.